In [1]:
#created by xihuanzhu at 9.27

In [2]:
import glob
import os
import time
import skimage.draw
import nibabel as nib
import cv2

import pydicom as dicom

# import pydicom as dicom
import numpy as np
import SimpleITK as sitk
from shapely.geometry.polygon import Polygon
import torch
from PIL import Image, ImageOps, ImageFilter
from DropBlock import DropBlock2D
from model.FPN import FPN

In [3]:
# model = FPN([2,4,23,3], 1, back_bone="resnet50")
#print(model)

In [4]:
# train_slice = np.load( "./npy_data/train_slice.npy")
# train_mask = np.load( "./npy_data/train_mask.npy")
# test_slice = np.load( "./npy_data/test_slice.npy")
# test_mask = np.load( "./npy_data/test_mask.npy")
# print(train_slice.shape)
# print(train_mask.shape)
# print(test_slice.shape)
# print(test_mask.shape)

In [5]:
# import matplotlib.pyplot as plt
# plt.imshow(np.squeeze(test_slice[0]), cmap ='gray')
# print(test_slice[0])

In [6]:
import random
SEED = 42

def seed_everything(seed):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

seed_everything(SEED)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
device_ids=range(torch.cuda.device_count())
device = torch.device("cuda:0")

In [7]:
import albumentations as A
# from custom_data_aug import *

def Get_Train_Transform():
#     return A.Compose([
#         RandomHorizontalFlip(),
#         RandomRotate(degree=30),
#         RandomGaussianBlur()
#     ])
   return MyTransform(degree=30)

class MyTransform(object):
    def __init__(self, degree):
        self.degree = degree

    def __call__(self, sample):

        #print("&&&&&&&&&", sample['image'].dtype, "*****", sample['image'].shape, "%%%%%%%%%")
        img =  np.squeeze(sample['image']).astype(np.uint8)
        mask = np.squeeze(sample['label']).astype(np.uint8)
        img = Image.fromarray(img)
        mask = Image.fromarray(mask)
        # # print(img.type)

        #RandomRotate
        rotate_degree = random.uniform(-1*self.degree, self.degree)
        img = img.rotate(rotate_degree, Image.BILINEAR)
        mask = mask.rotate(rotate_degree, Image.NEAREST)
        
        #RandomHorizontalFlip
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
            mask = mask.transpose(Image.FLIP_LEFT_RIGHT)
            
        #RandomGaussianBlur
        if random.random() < 0.5:
            img = img.filter(ImageFilter.GaussianBlur(
                radius=random.random()))

        img = np.array(img)
        mask = np.array(mask)
        img = img[np.newaxis, :, :]
        mask = mask[np.newaxis, :, :]

        return {'image': img, 'label': mask}

In [8]:
from albumentations import (
    PadIfNeeded,
    HorizontalFlip,
    VerticalFlip,    
    CenterCrop,    
    Crop,
    Compose,
    Transpose,
    RandomRotate90,
    ElasticTransform,
    GridDistortion, 
    OpticalDistortion,
    RandomSizedCrop,
    OneOf,
    CLAHE,
    RandomContrast,
    RandomGamma,
    CoarseDropout,
    RandomBrightness
)
def  augment_flips_color():
    return Compose([
        RandomSizedCrop(min_max_height=(256, 512), 
                           height=512, 
                           width=512, p=0.5),    
        VerticalFlip(p=0.5),              
        RandomRotate90(p=0.5),
        OneOf([ElasticTransform(p=0.5, 
                            alpha=120, 
                            sigma=120 * 0.05, 
                            alpha_affine=120 * 0.03),
        GridDistortion(p=0.5),
        OpticalDistortion(p=1, distort_limit=2, shift_limit=0.5)           
        ], p=0.8),
        #RandomContrast(p=0.8),
        #RandomBrightness(p=0.8),
        #RandomGamma(p=0.8),
        CoarseDropout(max_holes=16, max_height=16, max_width=16, p=0.7)])

aug = augment_flips_color()

In [9]:
from torch.utils.data import Dataset,DataLoader
from PIL import Image 
class DatasetRetriever(Dataset):
    def __init__(self, slice_path=None, mask_path=None, transforms=None):
        super().__init__()
        self.slice_path = slice_path
        self.mask_path = mask_path
        self.transforms = transforms
           
    def __getitem__(self, index):
        #print("***********************", index, "************************")
        image = np.load(f'{self.slice_path}/{str(index)}.npy')
        image = image.astype(np.float32)
        mask = np.load(f'{self.mask_path}/{str(index)}.npy')
        image= image.transpose(1, 2, 0)
        mask = mask.transpose(1, 2, 0)
        #print(image.dtype, "**", image.shape, "**", mask.shape)
        image = np.clip(image, -1000, 1000)
        #sample = {'image': image, 'label': mask}
        sample = self.transforms(image=image, mask=mask)
        image = sample['image'].transpose(2, 0, 1)
        mask = sample['mask'].transpose(2, 0 , 1)
        return image, mask
    
    def __len__(self):
        return len(os.listdir(self.slice_path))
    

In [10]:
import torch

transforms = aug
batch_size = 24
num_workers=2
train_slice_path = "/raid/zhuxihuan/data/body/MC_uncertainty_single_data/train_slice"
train_mask_path = "/raid/zhuxihuan/data/body/MC_uncertainty_single_data/train_mask"
test_slice_path = "/raid/zhuxihuan/data/body/single_npy_data/val_slice"
test_mask_path = "/raid/zhuxihuan/data/body/single_npy_data/val_mask"

train_dataset = DatasetRetriever(slice_path=train_slice_path, mask_path=train_mask_path, transforms=transforms)
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size = batch_size,
    pin_memory=True,
    drop_last=True,
    num_workers=num_workers
)
test_dataset = DatasetRetriever(slice_path=test_slice_path, mask_path=test_mask_path, transforms=transforms)
test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=4,
    pin_memory=True,
    drop_last=True,
    num_workers=1
)

In [11]:

k = np.load(train_mask_path + "/48.npy")
print(k.dtype)
k = k.astype(np.float32)
print(k.dtype)

float32
float32


In [12]:
print(len(test_loader))
print(len(train_loader))

345
232


In [13]:
import torch.nn as nn
import torch.nn.functional as F
class SoftDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(SoftDiceLoss, self).__init__()
        
    def forward(self, logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = torch.sigmoid(logits)
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = 1 - score.sum() / num
        return score

In [14]:
class Two_SoftDiceLoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(Two_SoftDiceLoss, self).__init__()
        
    def forward(self, logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = torch.sigmoid(logits)
        zero_probs  = 1 - probs
        zero_targets = 1 - targets
        
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        

        zero_m1 = zero_probs.view(num, -1)
        zero_m2 = zero_targets.view(num, -1)
        zero_intersection = (zero_m1 * zero_m2)
        zero_score = 2. * (zero_intersection.sum(1)) / (zero_m1.sum(1) + zero_m2.sum(1) + smooth)
        
        score = 2 - (score.sum() + zero_score.sum())/ num
        return score
    
class MyBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(MyBCELoss, self).__init__()
        
    def forward(self, logits, targets):
        
        #probs = torch.softmax(logits)
        #probs = torch.sigmoid(logits)
        
        return torch.nn.BCEWithLogitsLoss()(logits, targets)
    
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()
        
    def forward(self, logits, targets):
        return 0.5 * MyBCELoss()(logits, targets) + 0.5 * Two_SoftDiceLoss()(logits, targets)

In [15]:
  
# """ Parts of the U-Net model """

# import torch
# import torch.nn as nn
# import torch.nn.functional as F


# class DoubleConv(nn.Module):
#     """(convolution => [BN] => ReLU) * 2"""

#     def __init__(self, in_channels, out_channels, mid_channels=None):
#         super().__init__()
#         if not mid_channels:
#             mid_channels = out_channels
#         self.double_conv = nn.Sequential(
#             nn.Conv2d(in_channels, mid_channels, kernel_size=3, padding=1),
#             nn.BatchNorm2d(mid_channels),
#             nn.ReLU(inplace=True),
#             nn.Conv2d(mid_channels, out_channels, kernel_size=3, padding=1),
#             nn.BatchNorm2d(out_channels),
#             nn.ReLU(inplace=True)
#         )

#     def forward(self, x):
#         return self.double_conv(x)


# class Down(nn.Module):
#     """Downscaling with maxpool then double conv"""

#     def __init__(self, in_channels, out_channels):
#         super().__init__()
#         self.maxpool_conv = nn.Sequential(
#             nn.MaxPool2d(2),
#             DoubleConv(in_channels, out_channels)
#         )

#     def forward(self, x):
#         return self.maxpool_conv(x)


# class Up(nn.Module):
#     """Upscaling then double conv"""

#     def __init__(self, in_channels, out_channels, bilinear=True):
#         super().__init__()

#         # if bilinear, use the normal convolutions to reduce the number of channels
#         if bilinear:
#             self.up = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
#             self.conv = DoubleConv(in_channels, out_channels, in_channels // 2)
#         else:
#             self.up = nn.ConvTranspose2d(in_channels , in_channels // 2, kernel_size=2, stride=2)
#             self.conv = DoubleConv(in_channels, out_channels)


#     def forward(self, x1, x2):
#         x1 = self.up(x1)
#         # input is CHW
#         diffY = x2.size()[2] - x1.size()[2]
#         diffX = x2.size()[3] - x1.size()[3]

#         x1 = F.pad(x1, [diffX // 2, diffX - diffX // 2,
#                         diffY // 2, diffY - diffY // 2])
#         # if you have padding issues, see
#         # https://github.com/HaiyongJiang/U-Net-Pytorch-Unstructured-Buggy/commit/0e854509c2cea854e247a9c615f175f76fbb2e3a
#         # https://github.com/xiaopeng-liao/Pytorch-UNet/commit/8ebac70e633bac59fc22bb5195e513d5832fb3bd
#         x = torch.cat([x2, x1], dim=1)
#         return self.conv(x)


# class OutConv(nn.Module):
#     def __init__(self, in_channels, out_channels):
#         super(OutConv, self).__init__()
#         self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1)

#     def forward(self, x):
#         return self.conv(x)

# """ Full assembly of the parts to form the complete network """

# import torch.nn.functional as F

# # class UNet(nn.Module):
# #     def __init__(self, n_channels, n_classes, bilinear=True):
# #         super(UNet, self).__init__()
# #         self.n_channels = n_channels
# #         self.n_classes = n_classes
# #         self.bilinear = bilinear
# #         self.drop = nn.Dropout(0.1)

# #         self.inc = DoubleConv(n_channels, 64)
# #         self.down1 = Down(64, 128)
# #         self.down2 = Down(128, 256)
# #         factor = 2 if bilinear else 1
# #         #self.down3 = Down(256, 512 // factor)
# #         self.down3 = Down(256, 512)
# #         #factor = 2 if bilinear else 1
# #         self.down4 = Down(512, 1024 // factor)
# #         self.up1 = Up(1024, 512 // factor, bilinear)
# #         self.up2 = Up(512, 256 // factor, bilinear)
# #         self.up3 = Up(256, 128 // factor, bilinear)
# #         self.up4 = Up(128, 64, bilinear)
# #         self.outc = OutConv(64, n_classes)

# #     def forward(self, x):
# #         x1 = self.inc(x)
# #         x2 = self.down1(x1)
# #         x3 = self.down2(x2)
# #         x3 = self.drop(x3)
# #         x4 = self.down3(x3)
# #         x5 = self.down4(x4)
# #         x = self.up1(x5, x4)
# #         x = self.up2(x, x3)
# #         x = self.drop(x)
# #         x = self.up3(x, x2)
# #         x = self.up4(x, x1)
# #         logits = self.outc(x)
# #         return logits

# class UNet(nn.Module):
#     def __init__(self, n_channels, n_classes, bilinear=True):
#         super(UNet, self).__init__()
#         self.n_channels = n_channels
#         self.n_classes = n_classes
#         self.bilinear = bilinear
#         self.conv2_drop = DropBlock2D(0.60)

#         self.inc = DoubleConv(n_channels, 64)
#         self.down1 = Down(64, 128)
#         self.down2 = Down(128, 256)
#         factor = 2 if bilinear else 1
#         self.down3 = Down(256, 512)
#         factor = 2 if bilinear else 1
#         self.down4 = Down(512, 1024 // factor)
#         self.up1 = Up(1024, 512 // factor, bilinear)
#         self.up2 = Up(512, 256 // factor, bilinear)
#         self.up3 = Up(256, 128 // factor, bilinear)
#         self.up4 = Up(128, 64, bilinear)
#         self.outc = OutConv(64, n_classes)

#     def forward(self, x):
#         x1 = self.inc(x)
#         x2 = self.down1(x1)
#         x2 = self.conv2_drop(x2)
#         x3 = self.down2(x2)
#         x3 = self.conv2_drop(x3)
#         x4 = self.down3(x3)
#         x4 = self.conv2_drop(x4)
#         x5 = self.down4(x4)
#         x = self.up1(x5, x4)
#         x = self.conv2_drop(x)
#         x = self.up2(x, x3)
#         x = self.conv2_drop(x)
#         x = self.up3(x, x2)
#         x = self.conv2_drop(x)
#         x = self.up4(x, x1)
#         logits = self.outc(x)
#         return logits   
    
# netd = UNet(n_channels=1, n_classes=2).cuda()
# print(netd)
# # b = torch.tensor(torch.rand(3, 1, 512, 512)).to("cuda:0")
# # a= netd(b)
# # print(a)
# # print(a.shape)

In [16]:
class TrainGlobalConfig:
    num_workeres = 2
    batch_size = 4
    n_epochs = 150
    lr = 3e-4
    base_dir = "./88_aug_use_dropblock_multi_loss_FPN_DiceBCE_class_model"
    if not os.path.exists(base_dir):
        os.makedirs(base_dir)
    log_path = f'{base_dir}/log.txt'
#     if os.path.exists(log_path):
#         os.remove(log_path)
    
    SchedulerClass = torch.optim.lr_scheduler.ReduceLROnPlateau
    scheduler_params = dict(
        mode='min',
        factor=0.8,
        patience=2,
        verbose=False,
        threshold=1e-4,
        threshold_mode='abs',
        cooldown=0,
        min_lr=1e-8,
        eps=1e-8
    )

In [17]:
from torch import optim
from glob import glob
import time
class Model_Pipeline(object):
    def __init__(self, model, device, config):
        self.config = config
        self.epoch = 0 
        self.best_loss = 10**5
        self.model = model
        self.device = device
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=config.lr, weight_decay=0.01)
        self.scheduler = config.SchedulerClass(self.optimizer, **config.scheduler_params)
        self.criterion = DiceBCELoss().to(self.device)
        self.log(f'Pipeline prepared. Device is {self.device}')
        
    def fit(self, train_loader, test_loader):
        for e in range(self.config.n_epochs):
            loss = self.train_one_epoch(train_loader)
            self.log(f'[RESULT]: Train. Epoch: {self.epoch}, loss: {loss:.5f}')
            print(f'[RESULT]: Train. Epoch: {self.epoch}, loss: {loss:.5f}')  
            
            loss = self.val_one_epoch(test_loader)
            self.log(f'[RESULT]: Test. Epoch: {self.epoch}, loss: {loss:.5f}')
            print(f'[RESULT]: Test epoch: {self.epoch}, loss: {loss:.5f}') 
                
            if loss < self.best_loss:
                self.best_loss = loss
                self.save_model(f'{self.config.base_dir}/best-loss-{str(self.epoch).zfill(3)}epoch.bin')
                for path in sorted(glob(f'{self.config.base_dir}/best-loss-*epoch.bin'))[:-6]:
                    os.remove(path)
                        
            # scheduler todo
            self.epoch += 1
                
    def train_one_epoch(self, train_loader):
        self.model.train()
        summary_loss = 0.0
            
        #print(list(enumerate(train_loader)))
        for step, (images, labels) in enumerate(train_loader):
            print(f'Train Step {step}/{len(train_loader)}, ' + f'summary_loss: {summary_loss:.5f}', end='\r')
            labels = labels.to(self.device).float()
            images = images.to(self.device).float()
                
            self.optimizer.zero_grad()
            output = self.model(images)
            #print(output.shape)
#             t_loss = 0
#             for i in range(3):
#                 loss = self.criterion(output[i], labels)
#                 t_loss += loss
#             t_loss = t_loss/3.0

#             loss1 = self.criterion(output[0], labels)
#             #print(output[1].shape)
#             loss2 = torch.sum(output[1])/output[1].shape[0]/output[1].shape[2]/output[1].shape[3]
#             loss = loss1 * 0.9 +loss2 * 0.1
        
    
            loss = self.criterion(output[0], labels) + self.criterion(output[1], labels) \
                    + self.criterion(output[2], labels)
            
#             print(loss1)
#             print(loss2)
            loss = loss/3.0
            summary_loss += loss
            loss.backward()
                
            self.optimizer.step()
                
        return summary_loss/len(train_loader)
        
    def val_one_epoch(self, test_loader):
        self.model.eval()
        summary_loss = 0.0
            
        for step, (images, labels) in enumerate(test_loader):
            with torch.no_grad():
                print(f'Test Step {step}/{len(test_loader)}, ' + f'summary_loss: {summary_loss:.5f}', end='\r')
                labels = labels.to(self.device).float()
                images = images.to(self.device).float()
                output = self.model(images)
#                 t_loss = 0
#                 for i in range(3):
#                     loss = self.criterion(output[i], labels)
#                     t_loss += loss
#                 t_loss = t_loss/3.0
#                 summary_loss += t_loss

#                 loss1 = self.criterion(output[0], labels)
#                 #print(output[1].shape)
#                 loss2 = torch.sum(output[1])/output[1].shape[0]/output[1].shape[2]/output[1].shape[3]
#                 loss = loss1 * 0.9 +loss2 * 0.1

                
                loss = self.criterion(output[0], labels) + self.criterion(output[1], labels) \
                    + self.criterion(output[2], labels) 
            
#             print(loss1)
#             print(loss2)
                loss = loss/3.0
                summary_loss += loss
                    
        return summary_loss/len(test_loader)
        
    def log(self, message):
        with open(self.config.log_path, 'a+') as logger:
            logger.write(f'{message}\n')
                
    def save_model(self, path):
        self.model.eval()
        torch.save({
            'model_state_dict': self.model.state_dict(),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'scheduler_state_dict': self.scheduler.state_dict(),
            'best_score': self.best_loss
         }, path)
    

In [18]:
def main():
    input_channels = 1
    output_channels = 1
    model = FPN([2,4,23,3], output_channels, back_bone="resnet50", isTrained=True).cuda()
    #print(model)
#     input = torch.rand(10,1,512,512).cuda()
#     output = model(input)
#     print(output)
    filtter = Model_Pipeline(model=model, device=device, config=TrainGlobalConfig)
    filtter.fit(train_loader, test_loader)
#main()

[RESULT]: Train. Epoch: 0, loss: 0.2692001
[RESULT]: Test epoch: 0, loss: 0.17429079
[RESULT]: Train. Epoch: 1, loss: 0.1126429
[RESULT]: Test epoch: 1, loss: 0.09446809
[RESULT]: Train. Epoch: 2, loss: 0.0828580
[RESULT]: Test epoch: 2, loss: 0.09215111
[RESULT]: Train. Epoch: 3, loss: 0.0631620
[RESULT]: Test epoch: 3, loss: 0.05977133
[RESULT]: Train. Epoch: 4, loss: 0.0535713
[RESULT]: Test epoch: 4, loss: 0.05497766
[RESULT]: Train. Epoch: 5, loss: 0.0480016
[RESULT]: Test epoch: 5, loss: 0.05756328
[RESULT]: Train. Epoch: 6, loss: 0.0452619
[RESULT]: Test epoch: 6, loss: 0.04174390
[RESULT]: Train. Epoch: 7, loss: 0.040337
[RESULT]: Test epoch: 7, loss: 0.04702214
[RESULT]: Train. Epoch: 8, loss: 0.0444081
[RESULT]: Test epoch: 8, loss: 0.04774983
[RESULT]: Train. Epoch: 9, loss: 0.039071
[RESULT]: Test epoch: 9, loss: 0.05006077
[RESULT]: Train. Epoch: 10, loss: 0.03436
[RESULT]: Test epoch: 10, loss: 0.0408100
[RESULT]: Train. Epoch: 11, loss: 0.03216
[RESULT]: Test epoch: 11, 

[RESULT]: Train. Epoch: 98, loss: 0.01571
[RESULT]: Test epoch: 98, loss: 0.015437
[RESULT]: Train. Epoch: 99, loss: 0.01575
[RESULT]: Test epoch: 99, loss: 0.015444
[RESULT]: Train. Epoch: 100, loss: 0.01703
[RESULT]: Test epoch: 100, loss: 0.01680
[RESULT]: Train. Epoch: 101, loss: 0.01487
[RESULT]: Test epoch: 101, loss: 0.01512
[RESULT]: Train. Epoch: 102, loss: 0.01487
[RESULT]: Test epoch: 102, loss: 0.01739
[RESULT]: Train. Epoch: 103, loss: 0.01474
[RESULT]: Test epoch: 103, loss: 0.01442
[RESULT]: Train. Epoch: 104, loss: 0.01472
[RESULT]: Test epoch: 104, loss: 0.01526
[RESULT]: Train. Epoch: 105, loss: 0.01429
[RESULT]: Test epoch: 105, loss: 0.01508
[RESULT]: Train. Epoch: 106, loss: 0.01501
[RESULT]: Test epoch: 106, loss: 0.01737
[RESULT]: Train. Epoch: 107, loss: 0.01497
[RESULT]: Test epoch: 107, loss: 0.01436
[RESULT]: Train. Epoch: 108, loss: 0.01589
[RESULT]: Test epoch: 108, loss: 0.01648
[RESULT]: Train. Epoch: 109, loss: 0.01518
[RESULT]: Test epoch: 109, loss: 0.

In [19]:
#exitexixixixixiii
"""
----------------------------------------------
TEST
----------------------------------------------
"""

'\n----------------------------------------------\nTEST\n----------------------------------------------\n'

In [20]:
# class SoftDice(nn.Module):
#     def __init__(self, weight=None, size_average=True):
#         super(SoftDice, self).__init__()
        
#     def forward(self, logits, targets):
#         num = targets.size(0)
#         smooth = 0.000001
        
#         probs = F.sigmoid(logits)
#         m1 = probs.view(num, -1)
#         m2 = targets.view(num, -1)
#         intersection = (m1 * m2)
        
#         score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
#         score = score.sum() / num
#         return score
class CEToSoftDice(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(CEToSoftDice, self).__init__()
        
    def forward(self, logits, targets):
        #logits = torch.nn.functional.softmax(torch.tensor(A), dim=-3)
        num = targets.size(0)
        smooth = 0.000001
        
        probs = F.sigmoid(logits)
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = score.sum() / num
        return score

In [21]:
# os.environ["CUDA_VISIBLE_DEVICES"] = "2"
# device_ids=range(torch.cuda.device_count())
# device = torch.device("cuda:0")
import torch
class Model_Predict(object):
    def __init__(self, model, device, log_path):
        self.model = model
        self.device = device
        self.criterion = CEToSoftDice().to(self.device)
        self.loss = 0
        self.log_path = log_path
        for i in range(5):
            if os.path.exists(log_path + f"/log{i}.txt"):
                os.remove(log_path + f"/log{i}.txt")
        #self.mask = mask
        pass
    
    def val_one_epoch(self, i, test_loader):
        self.model.train()
        for mod in self.model.modules():
            if isinstance(mod, nn.GroupNorm) or isinstance(mod, nn.BatchNorm2d):
                mod.eval()
        summary_loss = 0
        mask = torch.Tensor(np.array([])).to(self.device)
#         img = torch.Tensor(np.array([])).to(self.device)
        #t_mask = torch.Tensor(np.array([])).to(self.device)
            
        for step, (images, labels) in enumerate(test_loader):
            with torch.no_grad():
                labels = labels.to(self.device).float()
                images = images.to(self.device).float()
                output = self.model(images)
                #output = torch.as_tensor(output)
               # print(output[1].shape)
                temp = output[0]
                for i in range(1, len(output)):
                    temp = torch.cat((temp, output[i]), dim=1)
                print(temp.shape)
                
                if len(mask):
                    mask = torch.cat((mask, temp))
                else:
                    mask = temp
                
                #print(mask.shape)
                for i in range(4):
                    loss = self.criterion(output[i], labels)
                    self.log(f"A single two-dimensional picture'dice is {loss}", i)
                    if i == 0:
                        summary_loss += loss
        self.log(f"A patient'dice is {summary_loss/len(test_loader)}", 0) 
        
        return summary_loss, mask
    
    def log(self, message, i):
        with open(self.log_path + f"/log{i}.txt", 'a+') as logger:
            logger.write(f'{message}\n')

def load(path):
    checkpoint = torch.load(path)
    output_channels  = 1
    model = FPN([2,4,23,3], output_channels, back_bone="resnet50", isTrained=False)
    model.load_state_dict(checkpoint['model_state_dict']) 
    return model

In [22]:
from torch.utils.data import Dataset,DataLoader
class DatasetRetrieverForOne(Dataset):
    def __init__(self, data_slice, data_mask, transforms=None):
        super().__init__()
        self.slicer = data_slice
        self.labels = data_mask
        self.transforms = transforms
           
    def __getitem__(self, index):
        """
        code for transform
        """
        return np.clip(self.slicer[index], -1000, 1000), self.labels[index]
    
    def __len__(self):
        return len(self.slicer)
    

In [23]:
def  tt_augment():
    return Compose([
        RandomSizedCrop(min_max_height=(256, 512), 
                           height=512, 
                           width=512, p=0.5),    
        VerticalFlip(p=0.5),              
        RandomRotate90(p=0.5),
        OneOf([ElasticTransform(p=0.5, 
                            alpha=120, 
                            sigma=120 * 0.05, 
                            alpha_affine=120 * 0.03),
        GridDistortion(p=0.5),
        OpticalDistortion(p=1, distort_limit=2, shift_limit=0.5)           
        ], p=0.8),
        #RandomContrast(p=0.8),
        #RandomBrightness(p=0.8),
        #RandomGamma(p=0.8),
        CoarseDropout(max_holes=16, max_height=16, max_width=16, p=0.7)])

aug = tt_augment()

In [24]:
# import torch
# #from npy2dcm import write_dicom

# transforms = None
# num_workers=2
# sample = 4

# #真实每个患者npy数据存储位置
# t_val_dir = "/raid/zhuxihuan/data/body/test_npy_data/"
# #存储预测的每个患者的mask的npy目录
# pre_val_dir = "/raid/zhuxihuan/data/body/88_aug_dropblock_FPN_DiceBCE_class_model_predict"
# if not os.path.exists(pre_val_dir):
#     os.makedirs(pre_val_dir)

# #存储预测的每个患者的mask的dcm文件目录
# save_pre_dicom = "/raid/zhuxihuan/data/body/two_pre_test5_for_val_dcm_dir"
# if not os.path.exists(save_pre_dicom):
#     os.makedirs(save_pre_dicom)
    
# def Get_Test_Result(sample, val_dir, pre_val_dir):   
#     for sdir in os.listdir(t_val_dir)[:1]:
#         f_dir = os.path.join(t_val_dir, sdir)
#         t_slice = np.load(f_dir + "/slice.npy")
#         t_mask = np.load(f_dir + "/mask.npy")
#         print(t_slice.shape)
    
#         test_dataset = DatasetRetrieverForOne(data_slice=t_slice, data_mask=t_mask, transforms=transforms)
#         test_loader = torch.utils.data.DataLoader(
#             test_dataset,
#             batch_size=1,
#             pin_memory=True,
#             drop_last=True,
#             num_workers=1
#         )
    
#         base_path = os.path.join(pre_val_dir, sdir)
#         if not os.path.exists(base_path):
#             os.makedirs(base_path)
#         dcm_base_path = os.path.join(save_pre_dicom, sdir)
#         if not os.path.exists(dcm_base_path):
#             os.makedirs(dcm_base_path)
#         for j in range(2, 6):    
#             path = sorted(glob(f'60_aug_use_dropblock_FPN_DiceBCE_class_model/best-loss-*epoch.bin'))[-j]
#             net = load(path).cuda()
#             for i in range(sample):
#                 print(f"/log{i + (j - 1) * sample}.txt")
#                 pre = Model_Predict(model=net, device=device, log_path=base_path+f"/log{i + (j - 1) * sample}.txt")
#                 _, pre_mask= pre.val_one_epoch(i, test_loader)
        
#                 pre_mask = torch.sigmoid(pre_mask)
#                 pre_mask = pre_mask.cpu()
            
#                 np.save(base_path + f"/pmask{i + (j - 1) * sample}.npy", pre_mask)
#                 print(pre_mask.dtype)
#                 pre_mask = pre_mask.numpy().transpose(0, 2, 3, 1).astype(np.uint8)
#                 print(pre_mask.shape, "*****")
            
#        # np.save(base_path + f"/origin_mask{i}.npy", pre_mask)
#         t_slice = np.squeeze(t_slice)
#         print(t_slice.shape)
        
#         #write_dicom(t_slice, pre_mask, dcm_base_path, roi_names=['body'], patient_id=sdir, pixel_spacing=(1.0, 1.0, 1.0))
#         #print(pre_mask.shape)
#         #return pre_mask
# pre_mk = np.array([])       
# Get_Test_Result(sample, t_val_dir, pre_val_dir)
#     #np.append(pre_mk, temp)

# #np.save(base_path + "/pmask.npy", pre_mask)

# # for image, label in train_loader:
# #         print(image.shape)
# # print("*****")
# # for image, label in test_loader:14444444444
# #         print(image.shape)

In [25]:
import torch
#from npy2dcm import write_dicom

transforms = None
num_workers=2
sample = 4

#真实每个患者npy数据存储位置
t_val_dir = "/raid/zhuxihuan/data/body/test_npy_data/"
#存储预测的每个患者的mask的npy目录
pre_val_dir = "/raid/zhuxihuan/data/body/88_aug_dropblock_FPN_DiceBCE_class_model_predict"
if not os.path.exists(pre_val_dir):
    os.makedirs(pre_val_dir)

#存储预测的每个患者的mask的dcm文件目录
save_pre_dicom = "/raid/zhuxihuan/data/body/two_pre_test5_for_val_dcm_dir"
if not os.path.exists(save_pre_dicom):
    os.makedirs(save_pre_dicom)
    
def Get_Test_Result(sample, val_dir, pre_val_dir):   
    for sdir in os.listdir(t_val_dir)[:1]:
        f_dir = os.path.join(t_val_dir, sdir)
        t_slice = np.load(f_dir + "/slice.npy")
        t_mask = np.load(f_dir + "/mask.npy")
        print(t_slice.shape)
    
        test_dataset = DatasetRetrieverForOne(data_slice=t_slice, data_mask=t_mask, transforms=transforms)
        test_loader = torch.utils.data.DataLoader(
            test_dataset,
            batch_size=1,
            pin_memory=True,
            drop_last=True,
            num_workers=1
        )
    
        base_path = os.path.join(pre_val_dir, sdir)
        if not os.path.exists(base_path):
            os.makedirs(base_path)
        dcm_base_path = os.path.join(save_pre_dicom, sdir)
        if not os.path.exists(dcm_base_path):
            os.makedirs(dcm_base_path)
#         for j in range(2, 6):    
        path = sorted(glob(f'88_aug_use_dropblock_multi_loss_FPN_DiceBCE_class_model/best-loss-*epoch.bin'))[-1]
        net = load(path).cuda()
           # for i in range(sample):
        #print(f"/log{i + (j - 1) * sample}.txt")
        pre = Model_Predict(model=net, device=device, log_path=base_path)
        _, pre_mask= pre.val_one_epoch(1, test_loader)
        print(pre_mask.shape)
        for level in range(4):
            ppre_mask = torch.sigmoid(pre_mask[:, level, :])
            ppre_mask = ppre_mask.cpu()

            np.save(base_path + f"/pmask{level}.npy", ppre_mask)
            print(ppre_mask.dtype)
#             ppre_mask = pre_mask.numpy().transpose(0, 2, 3, 1).astype(np.uint8)
#             print(ppre_mask.shape, "*****")
            
       # np.save(base_path + f"/origin_mask{i}.npy", pre_mask)
        t_slice = np.squeeze(t_slice)
        print(t_slice.shape)
        
        #write_dicom(t_slice, pre_mask, dcm_base_path, roi_names=['body'], patient_id=sdir, pixel_spacing=(1.0, 1.0, 1.0))
        #print(pre_mask.shape)
        #return pre_mask
pre_mk = np.array([])       
Get_Test_Result(sample, t_val_dir, pre_val_dir)
    #np.append(pre_mk, temp)

#np.save(base_path + "/pmask.npy", pre_mask)

# for image, label in train_loader:
#         print(image.shape)
# print("*****")
# for image, label in test_loader:
#         print(image.shape)

(263, 1, 512, 512)
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])


/usr/local/anaconda3/lib/python3.6/site-packages/torch/nn/functional.py:1569: UserWarning: nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.
  warnings.warn("nn.functional.sigmoid is deprecated. Use torch.sigmoid instead.")


torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1, 4, 512, 512])
torch.Size([1,

In [26]:
file_dir = "/raid/zhuxihuan/data/body/88_aug_dropblock_FPN_DiceBCE_class_model_predict/"
for sdir in os.listdir(file_dir):
    path = os.path.join(file_dir, sdir) + "/"
    print(path)
    pmk = []
    for s in range(0, 4):
        print(s)
        temp = np.load(path + f"pmask{s}.npy")
        print(temp.shape)
        pmk.append(temp)
    print(len(pmk))
    mean_pmk = np.array(pmk)
    print(mean_pmk.shape)
    mean_pmk = mean_pmk.mean(axis=0)
    print(np.array(mean_pmk).shape)
    np.save(path + "mean_mask.npy", (mean_pmk))
    var_pmk = np.array(pmk)
    var_pmk = var_pmk.var(axis=0)
    print(var_pmk.shape)
    np.save(path + "var_mask.npy", var_pmk)
# test_path = "/raid/zhuxihuan/data/body/mc_dropout_BCE_pre_test5_dir/BodyV2_17-BodyV2_17/"
# pmk = []
# for s in range(sample * 4):
#     temp = np.load(test_path + f"pmask{s}.npy")
#     print(temp.shape)
#     pmk.append(temp)
# print(len(pmk))
# mean_pmk = np.array(pmk)
# print(mean_pmk.shape)
# mean_pmk = mean_pmk.mean(axis=0)
# print(np.array(mean_pmk).shape)
# np.save(test_path + "mean_mask.npy", (mean_pmk))
# var_pmk = np.array(pmk)
# var_pmk = var_pmk.var(axis=0)
# print(var_pmk.shape)
# np.save(test_path + "var_mask.npy", var_pmk)

/raid/zhuxihuan/data/body/88_aug_dropblock_FPN_DiceBCE_class_model_predict/BodyV2_17-BodyV2_17/
0
(263, 512, 512)
1
(263, 512, 512)
2
(263, 512, 512)
3
(263, 512, 512)
4
(4, 263, 512, 512)
(263, 512, 512)
(263, 512, 512)


In [27]:
tets

hh = np.load("/raid/zhuxihuan/data/body/mc_aug_dropblock88_BCE_pre_test5_dir/BodyV2_17-BodyV2_17/pmask0.npy")
print(hh.shape)

NameError: name 'tets' is not defined

In [ ]:
A = np.random.randn(2, 2, 3,  3)
print(A)
c = torch.nn.functional.softmax(torch.tensor(A), dim=-3)
print(c)
k = torch.argmax(c, dim=-3)
print(k)

In [ ]:
def forward(logits, targets):
        num = targets.size(0)
        smooth = 0.000001
        
        probs = logits
        m1 = probs.view(num, -1)
        m2 = targets.view(num, -1)
        intersection = (m1 * m2)
        
        score = 2. * (intersection.sum(1)) / (m1.sum(1) + m2.sum(1) + smooth)
        score = score.sum() / num
        return score
A = np.array([[[[0.1,0.2],[0.5,0.4]]],
             [[[0.1,0.8],[0.6,0.4]]]])
C = np.array([[[[0.,0.],[1.,0.]]],
             [[[0.,1.],[1.,0.]]]])
B = np.array([[[[0,0],[0,0]]],
             [[[0,1],[0,0]]]])
print(forward(torch.tensor(A), torch.tensor(B)))
print(forward(torch.tensor(C), torch.tensor(B)))

In [ ]:
print(0.1*np.log(0.9))